# Section 7 · Observability 3 — Langfuse

[Langfuse](https://langfuse.com) is an open-source LLM observability platform: cost tracking, prompt versioning, scoring and nested trace views. Use the managed cloud or self-host it.

**Get your keys**
1. Sign up at https://cloud.langfuse.com (EU) or https://us.cloud.langfuse.com (US)
2. Create a project → **Settings → API Keys → Create new API keys**
3. Copy the **Public Key** (`pk-lf-...`) and **Secret Key** (`sk-lf-...`)

The Langfuse SDK registers itself as the OpenTelemetry tracer provider, so once the client is initialised, Strands spans flow to Langfuse automatically.

> Each tracing notebook should run in a **fresh kernel**: OpenTelemetry allows only one global tracer provider per Python process.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import os

MODEL_ID = os.getenv("BEDROCK_MODEL_ID", "global.anthropic.claude-sonnet-4-6")
REGION = os.getenv("AWS_REGION", os.getenv("AWS_DEFAULT_REGION", "us-east-1"))
os.environ.setdefault("AWS_REGION", REGION)

In [ ]:
# Your Langfuse project keys (or export them in your shell before starting Jupyter)
os.environ.setdefault("LANGFUSE_PUBLIC_KEY", "pk-lf-...")
os.environ.setdefault("LANGFUSE_SECRET_KEY", "sk-lf-...")
os.environ.setdefault("LANGFUSE_BASE_URL", "https://cloud.langfuse.com")  # US: https://us.cloud.langfuse.com

In [ ]:
from langfuse import get_client

langfuse = get_client()

if langfuse.auth_check():
    print("Langfuse client is authenticated and ready.")
else:
    print("Authentication failed. Check your keys and base URL.")

In [ ]:
from strands import Agent
from strands.models.bedrock import BedrockModel

model = BedrockModel(model_id=MODEL_ID, region_name=REGION)
agent = Agent(
    model=model,
    system_prompt="You are a document analyst. Summarize documents clearly and concisely.",
    trace_attributes={"session.id": "langfuse-demo-session", "user.id": "learner@example.com"},
)

response = agent("What are the key risks in a standard SaaS subscription agreement?")

# Ensure traces are flushed to Langfuse before the notebook moves on
langfuse.flush()

Open your Langfuse project → **Tracing**. You'll see the system prompt and user message, each model call with tokens and latency, any tool calls, and an estimated cost.